In [2]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [ ]:
#Home Team
netherlands_world_cup_starting_xi_vs_japan = [
    ["Bart Verbruggen", "GK", "Brighton", 79],
    ["Denzel Dumfries", "RB", "Inter Milan", 83],
    ["Virgil van Dijk", "CB", "Liverpool", 87],
    ["Jan Paul van Hecke", "CB", "Brighton & Hove Albion", 81],
    ["Micky van de Ven", "LB", "Tottenham Hotspur", 82],
    ["Frenkie de Jong", "CDM", "Barcelona", 85],
    ["Ryan Gravenberch", "CDM", "Liverpool", 84],  
    ["Cody Gakpo", "LW", "Liverpool", 81],
    ["Crysencio Summerville", "RW", "West Ham United", 80],
    ["Tijjani Reijnders", "CAM", "Manchester City", 83],
    ["Donyell Malen", "ST", "AS Roma", 82],
    
]
netherlands_world_cup_bench_vs_japan= [
    ["Nathan Ake", "LB", "Manchester City", 79],
    ["Teun Koopmeiners", "CM", "Juventus", 78],
    ["Quinten Timber", "CM", "Marseille", 79,
    ["Memphis Depay", "ST", "Corinthians", 80],
    ["Brian Brobbey", "ST", "Sunderland", 78]

]


In [8]:
gk = GoalkeeperProfile("Verbruggen", "Netherlands", "goalkeeper", 79)
gk.input_match_stats(
    minutes_played=90,
    saves=1,
    saves_inside_box=0,
    saves_outside_box=1,
    goals_conceded=2,
    xG_faced=1.32,
    goals_prevented=-0.68,
    total_passes=30,
    accurate_passes=20,
    total_long_balls=14,
    accurate_long_balls=4,
    touches=33,
    errors=1,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Verbruggen's match rating: 5.79


In [10]:
player = PlayerProfile("Dumfries", "Netherlands", "defender", 83)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=33,
    total_passes=38,
    expected_goals=0,
    expected_assists=0.17,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=4,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=3,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Dumfries's match rating: 7.48


In [12]:
player = PlayerProfile("Van Dijk", "Netherlands", "defender", 87)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=95,
    total_passes=103,
    expected_goals=0.06,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=5,
    total_long_balls=9,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=8,
    interceptions=1,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=3,
    aerial_duels_total=3,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Van Dijk's match rating: 8.50


In [14]:
player = PlayerProfile("Van Hecke", "Netherlands", "defender", 81)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=87,
    total_passes=95,
    expected_goals=0.12,
    expected_assists=0.06,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=5,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=3,
    ground_duels_total=3,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Van Hecke's match rating: 7.40


In [16]:
player = PlayerProfile("Van de Ven", "Netherlands", "defender", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=41,
    total_passes=42,
    expected_goals=0.02,
    expected_assists=0.03,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=4,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=4,
    duels_lost=1,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=2,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Van de Ven's match rating: 6.50


In [18]:
player = PlayerProfile("De Jong", "Netherlands", "midfielder", 85)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=69,
    total_passes=73,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=2,
    interceptions=2,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=5,
    duels_lost=3,
    ground_duels_won=5,
    ground_duels_total=8,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

De Jong's match rating: 8.47


In [20]:
player = PlayerProfile("Gravenberch", "Netherlands", "midfielder", 84)

player.input_match_stats(
    minutes=81,
    goals=0,
    assists=2,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=22,
    total_passes=25,
    expected_goals=0,
    expected_assists=0.1,
    successful_dribbles=2,
    total_dribbles=3,
    accurate_crosses=1,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=3,
    duels_lost=3,
    ground_duels_won=2,
    ground_duels_total=5,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gravenberch's match rating: 8.94


In [22]:
player = PlayerProfile("Reijnders", "Netherlands", "midfielder", 83)

player.input_match_stats(
    minutes=70,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=19,
    total_passes=22,
    expected_goals=0,
    expected_assists=0.13,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=3,
    total_crosses=7,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=2,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Reijnders's match rating: 6.38


In [24]:
player = PlayerProfile("Summerville", "Netherlands", "fowrard", 80)

player.input_match_stats(
    minutes=70,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=25,
    total_passes=29,
    expected_goals=0.02,
    expected_assists=0.25,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=1,
    duels_won=5,
    duels_lost=2,
    ground_duels_won=4,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=3,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=True, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Summerville's match rating: 9.21


In [26]:
player = PlayerProfile("Gakpo", "Netherlands", "forward", 81)

player.input_match_stats(
    minutes=85,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=2,
    accurate_passes=27,
    total_passes=36,
    expected_goals=0.2,
    expected_assists=0.13,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=7,
    duels_lost=3,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=4,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gakpo's match rating: 7.67


In [28]:
player = PlayerProfile("Malen", "Netherlands", "forward", 82)

player.input_match_stats(
    minutes=70,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=2,
    accurate_passes=4,
    total_passes=8,
    expected_goals=0.21,
    expected_assists=0.34,
    successful_dribbles=2,
    total_dribbles=5,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=4,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=5,
    aerial_duels_won=2,
    aerial_duels_total=3,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Malen's match rating: 6.00


In [30]:
player = PlayerProfile("Depay", "Netherlands", "forward", 80)

player.input_match_stats(
    minutes=20,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=7,
    total_passes=7,
    expected_goals=0,
    expected_assists=0.12,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=0,
    duels_lost=3,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Depay's match rating: 5.85


In [34]:
player = PlayerProfile("Brobbey", "Netherlands", "forward", 78)

player.input_match_stats(
    minutes=5,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Brobbey's match rating: 6.00


In [36]:
player = PlayerProfile("Q. Timber", "Netherlands", "midfielder", 79)

player.input_match_stats(
    minutes=20,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=4,
    total_passes=5,
    expected_goals=0,
    expected_assists=0.22,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Q. Timber's match rating: 7.39


In [38]:
player = PlayerProfile("Koopmeiners", "Netherlands", "midfielder", 78)

player.input_match_stats(
    minutes=20,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=0,
    accurate_passes=1,
    total_passes=1,
    expected_goals=0.16,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Koopmeiners's match rating: 6.07


In [40]:
player = PlayerProfile("Ake", "Netherlands", "defender", 79)

player.input_match_stats(
    minutes=9,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=9,
    total_passes=10,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ake's match rating: 6.35


In [ ]:
#AwayTeam
japan_world_cup_starting_xi_vs_netherlands = [
    ["Zion Suzuki", "GK", "Parma Calcio", 79],
    ["Shogo Taniguchi", "CB", "Sint-Truiden", 74],   
    ["Hiroki Ito", "CB", "Bayern Munich", 77],
    ["Tsuyoshi Watanabe", "CB", "Feyenoord", 77],
    ["Kaishu Sano", "CM", "Mainz 05", 77],
    ["Daichi Kamada", "CM", "Crystal Palace", 78],
    ["Keito Nakamura", "LW", "Stade de Reims", 77],
    ["Takefusa Kubo", "CAM", "Real Sociedad", 80],
    ["Daizen Maeda", "CAM", "Celtic", 76],
    ["Ritsu Doan", "RW", "Eintracht Frankfurt", 80],
    ["Ayase Ueda", "ST", "Feyenoord", 76],
    

]
japan_world_cup_bench_vs_netherlands = [
    ["Junya Ito", "CAM", "Genk", 78],
    ["Takehiro Tomiyasu", "CB", "Ajax", 76],
    ["Yukinari Sugawara", "RW", "Werder Bremen", 75],
    ["Kento Shiode", "ST", "Wolfsburg", 72],
    ["Koki Ogawa", "ST", "NEC Nijmegen" , 74],
    

]   

In [42]:
gk = GoalkeeperProfile("Suzuki", "Japan", "goalkeeper", 79)
gk.input_match_stats(
    minutes_played=90,
    saves=4,
    saves_inside_box=4,
    saves_outside_box=0,
    goals_conceded=2,
    xG_faced=2.16,
    goals_prevented=0.16,
    total_passes=20,
    accurate_passes=10,
    total_long_balls=11,
    accurate_long_balls=1,
    touches=28,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Suzuki's match rating: 8.00


In [44]:
player = PlayerProfile("Watanabe", "Japan", "defender", 77)

player.input_match_stats(
    minutes=75,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=21,
    expected_goals=0,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=7,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=6,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=5,
    duels_lost=5,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=2,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Watanabe's match rating: 7.25


In [46]:
player = PlayerProfile("Taniguchi", "Japan", "defender", 74)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=49,
    total_passes=50,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=9,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Taniguchi's match rating: 8.22


In [50]:
player = PlayerProfile("Ito", "Japan", "defender", 77)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=37,
    total_passes=41,
    expected_goals=0.03,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=5,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=1,
    duels_lost=6,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ito's match rating: 5.85


In [52]:
player = PlayerProfile("Doan", "Japan", "midfelder", 80)

player.input_match_stats(
    minutes=75,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=11,
    total_passes=14,
    expected_goals=0,
    expected_assists=0.1,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=1,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Doan's match rating: 5.91


In [56]:
player = PlayerProfile("Sano", "Japan", "midfielder", 77)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=25,
    total_passes=31,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=2,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=2,
    duels_won=3,
    duels_lost=8,
    ground_duels_won=2,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sano's match rating: 5.90


In [58]:
player = PlayerProfile("Kamada", "Japan", "midfielder", 78)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=48,
    total_passes=59,
    expected_goals=0.12,
    expected_assists=0.12,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=3,
    total_long_balls=6,
    dispossessed=1,
    tackles_won=3,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=4,
    dribbled_past=2,
    duels_won=4,
    duels_lost=5,
    ground_duels_won=4,
    ground_duels_total=8,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kamada's match rating: 9.14


In [60]:
player = PlayerProfile("Nakamura", "Japan", "midfielder", 77)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=3,
    shots_on_target=1,
    accurate_passes=28,
    total_passes=31,
    expected_goals=0.08,
    expected_assists=0.04,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=5,
    accurate_long_balls=3,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Nakamura's match rating: 9.15


In [62]:
player = PlayerProfile("Maeda", "Japan", "forward", 76)

player.input_match_stats(
    minutes=66,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=4,
    total_passes=5,
    expected_goals=0,
    expected_assists=0.11,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=1,
    duels_lost=6,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Maeda's match rating: 5.65


In [64]:
player = PlayerProfile("Kubo", "Japan", "forward", 80)

player.input_match_stats(
    minutes=75,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=16,
    expected_goals=0.03,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kubo's match rating: 7.06


In [66]:
player = PlayerProfile("Ueda", "Japan", "forard", 76)

player.input_match_stats(
    minutes=84,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=4,
    total_passes=5,
    expected_goals=0.1,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=3,
    duels_lost=5,
    ground_duels_won=3,
    ground_duels_total=5,
    aerial_duels_won=0,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ueda's match rating: 5.95


In [68]:
player = PlayerProfile("Ito", "Japan", "fowrard", 78)

player.input_match_stats(
    minutes=24,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=10,
    total_passes=14,
    expected_goals=0,
    expected_assists=0.09,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=3,
    total_crosses=6,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ito's match rating: 6.70


In [70]:
player = PlayerProfile("Tomiyasu", "Japan", "defender", 76)

player.input_match_stats(
    minutes=15,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=19,
    total_passes=20,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Tomiyasu's match rating: 6.35


In [72]:
player = PlayerProfile("Suguwara", "midfielder", "Japan", 75)

player.input_match_stats(
    minutes=15,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=11,
    total_passes=12,
    expected_goals=0.16,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Suguwara's match rating: 7.80


In [74]:
player = PlayerProfile("Ogawa", "Japan", "forward", 74)

player.input_match_stats(
    minutes=15,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ogawa's match rating: 7.00


In [76]:
player = PlayerProfile("Shiogai", "Japan", "forward", 72)

player.input_match_stats(
    minutes=6,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=1,
    total_passes=1,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Shiogai's match rating: 6.45
